# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vagisha14/Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Two Paper Findings + My Methodology Questions

Finding 1:
The paper suggests that content performance and search-related signals can be used to identify content opportunities that deserve further review. This supports the use of measurable signals such as impressions, clicks, CTR, average position, engagement, content age, and recent performance in the content refresh scoring workflow.

Methodology question:
Do these signals provide enough evidence to reliably prioritize content refresh opportunities, or could the ranking be strongly affected by differences in traffic, search demand, content age, or other factors?

Finding 2:
The paper emphasizes that ranked recommendations should be treated as decision-support signals rather than guaranteed predictions of future performance. This supports validating the scoring approach against a baseline and checking for leakage before making strong claims.

Methodology question:
Does the Random Forest model provide useful improvement over the Week-4 baseline under an honest evaluation setup, and are the model's recommendations stable enough to support practical content review?

These findings and questions guide the validation audit and help ensure that the final claims are supported by the available evidence.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [2]:
# W06 Section 2 — Honest split validation

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load the dataset
url = "https://raw.githubusercontent.com/Vagisha14/Internship/main/data/raw/content_refresh_anonymized.csv"
data = pd.read_csv(url)

# Features used by the W05 model
feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct"
]

feature_cols = [c for c in feature_cols if c in data.columns]

X = data[feature_cols].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median(numeric_only=True))

# Same baseline objective used in W05
y = (
    0.30 * (1 - data["ctr"].rank(pct=True))
    + 0.25 * data["content_age_days"].rank(pct=True)
    + 0.25 * data["days_since_last_update"].rank(pct=True)
    + 0.20 * data["avg_position"].rank(pct=True)
)

valid = X.notna().all(axis=1) & y.notna()

X = X.loc[valid]
y = y.loc[valid]

# Sort by content age to create an honest time-like split
order = data.loc[valid, "content_age_days"].sort_values().index

X_ordered = X.loc[order]
y_ordered = y.loc[order]

split_point = int(len(X_ordered) * 0.80)

X_train = X_ordered.iloc[:split_point]
X_test = X_ordered.iloc[split_point:]

y_train = y_ordered.iloc[:split_point]
y_test = y_ordered.iloc[split_point:]

# Train model
honest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

# Predictions
honest_predictions = honest_model.predict(X_test)

# Metrics
honest_mae = mean_absolute_error(y_test, honest_predictions)
honest_rmse = np.sqrt(
    mean_squared_error(y_test, honest_predictions)
)

print("Honest split validation completed.")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("MAE:", round(honest_mae, 4))
print("RMSE:", round(honest_rmse, 4))

# Before/after comparison
before_after = pd.DataFrame({
    "Evaluation": ["W05 Random Split", "W06 Honest Split"],
    "MAE": [mae, honest_mae],
    "RMSE": [rmse, honest_rmse]
})

print("\nBefore / After Comparison:")
display(before_after)

Honest split validation completed.
Training rows: 24000
Testing rows: 6000
MAE: 0.0285
RMSE: 0.0348


NameError: name 'mae' is not defined

In [3]:
# W06 Section 2 — My model under an honest split

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load dataset
url = "https://raw.githubusercontent.com/Vagisha14/Internship/main/data/raw/content_refresh_anonymized.csv"
data = pd.read_csv(url)

# Model features
feature_cols = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "trend_pct"
]

feature_cols = [c for c in feature_cols if c in data.columns]

X = data[feature_cols].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median())

# Same target used in W05
y = (
    0.30 * (1 - data["ctr"].rank(pct=True))
    + 0.25 * data["content_age_days"].rank(pct=True)
    + 0.25 * data["days_since_last_update"].rank(pct=True)
    + 0.20 * data["avg_position"].rank(pct=True)
)

# Remove invalid rows
valid = X.notna().all(axis=1) & y.notna()

X = X.loc[valid]
y = y.loc[valid]

# Honest ordered split
# Earlier observations = training
# Later observations = testing
order = data.loc[valid].sort_values("content_age_days").index

X = X.loc[order]
y = y.loc[order]

split_point = int(len(X) * 0.80)

X_train = X.iloc[:split_point]
X_test = X.iloc[split_point:]

y_train = y.iloc[:split_point]
y_test = y.iloc[split_point:]

# Train Random Forest
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)

# Calculate metrics directly in this notebook
honest_mae = mean_absolute_error(y_test, predictions)
honest_rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("Honest split validation completed.")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Honest Split MAE:", round(honest_mae, 4))
print("Honest Split RMSE:", round(honest_rmse, 4))

# Simple baseline for the same test set:
# predict the training-set mean for every test observation
baseline_predictions = np.full(
    len(y_test),
    y_train.mean()
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)

# Comparison table
comparison = pd.DataFrame({
    "Method": [
        "Simple Baseline",
        "Random Forest - Honest Split"
    ],
    "MAE": [
        baseline_mae,
        honest_mae
    ],
    "RMSE": [
        baseline_rmse,
        honest_rmse
    ]
})

print("\nHonest Split Comparison:")
display(comparison)

Honest split validation completed.
Training rows: 24000
Testing rows: 6000
Honest Split MAE: 0.0285
Honest Split RMSE: 0.0348

Honest Split Comparison:


,Method,MAE,RMSE
0,Simple Baseline,0.166436,0.198656
1,Random Forest - Honest Split,0.028535,0.034757


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
# W06 Section 3 — Leakage Audit

# Columns that should NOT be used as model features
# because they are identifiers, precomputed decisions, or potential leakage fields.

suspicious_columns = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "impression_tier",
    "position_tier",
    "age_tier",
    "age_tier_order",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier"
]

# Check which suspicious columns actually exist
found_suspicious = [
    col for col in suspicious_columns
    if col in feature_cols
]

# Check the final feature set
leakage_audit = pd.DataFrame({
    "Column": feature_cols,
    "Potential_Leakage": [
        "YES" if col in found_suspicious else "NO"
        for col in feature_cols
    ]
})

print("=== LEAKAGE AUDIT ===")

if found_suspicious:
    print("\nPotentially problematic columns found:")
    for col in found_suspicious:
        print("-", col)
else:
    print("\nNo suspicious identifier/tier columns are being used.")

print("\nFinal model features:")
display(leakage_audit)

print("\nLeakage audit conclusion:")
print(
    "The model should exclude identifiers, precomputed tier fields, "
    "and any fields that directly encode the recommendation or outcome. "
    "The remaining features are used as predictive signals."
)

=== LEAKAGE AUDIT ===

No suspicious identifier/tier columns are being used.

Final model features:


,Column,Potential_Leakage
0,search_volume,NO
1,competition,NO
2,cpc,NO
3,word_count,NO
4,char_count,NO
5,impressions_90d,NO
6,clicks_90d,NO
7,pageviews_90d,NO
8,sessions_90d,NO
9,users_90d,NO



Leakage audit conclusion:
The model should exclude identifiers, precomputed tier fields, and any fields that directly encode the recommendation or outcome. The remaining features are used as predictive signals.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Claims Rewrite

The analysis provides evidence that a Random Forest model can learn patterns from content-performance and search-related signals and produce predictions for content opportunity scoring.

However, the results should not be interpreted as proof that the model will improve future content performance. The model was evaluated using an honest holdout split, and its performance should be compared with a simple baseline before making claims about improvement.

The scoring and model outputs should therefore be described as prioritization and decision-support signals rather than guaranteed predictions. The recommendations identify content items that may deserve review based on the available signals, but human review and further validation are required before taking action.

The analysis also has limitations related to the available features, data quality, historical observations, and potential changes in search or content behavior over time. Future testing on new observations would provide stronger evidence about whether the approach generalizes to unseen data.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.